In [ ]:
# youtube - https://www.youtube.com/watch?v=zCEJurLGFRk&t=14s
# % pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib gspread
import gspread
from google.oauth2.service_account import Credentials
import pandas as pd
from gspread_dataframe import set_with_dataframe

from data import Data
from cache_pandas import timed_lru_cache
import logging

@timed_lru_cache(seconds=None, maxsize=None)
def dataF():
    logging.info("Fetching data in dataF()")
    df = Data()
    vix_data = df.vix_history()
    policy_rate1, policy_rate2, policy_rate3 = df.policy_rate()
    fx = df.forex_exchange()
    cds = df.cds()
    liquidity = df.liquidity()
    gdp_growth = df.gdp_growth()
    logging.info("Completed fetching data in dataF()")
    return (
        vix_data, #use this
        policy_rate1, #use this
        policy_rate2,
        policy_rate3,
        fx, #use this
        cds, #use this
        liquidity, #use this
        gdp_growth, #use this
    )


vix2, policy_rate12, policy_rate22, policy_rate32, fx2, cds2, liquidity2, gdp_growth2, = dataF()

scopes = [
    'https://www.googleapis.com/auth/spreadsheets']

creds = Credentials.from_service_account_file(
    'credential.json', scopes=scopes)

client = gspread.authorize(creds)

# 1PP6gpBcoOHjjgCx7LuHLa3dv4ET6ufKvpSY4UDvBczQ
sheet_id = '11Ora6_5EoQJdgnUpjjZgFZyrILguo1c32mde_uQwupw'
sheet = client.open_by_key(sheet_id)

df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie"],
    "Age": [25, 30, 40]
})

set_with_dataframe(sheet.sheet1, df) 

values_list = sheet.sheet1.get_all_values()
print(values_list)

worksheet_list = map(lambda x: x.title, sheet.worksheets())
worksheet_name = {'vix_data': vix_data, 'policy_rate1': policy_rate1, 'fx': fx, 'cds': cds, 'liquidity': liquidity, 'gdp_growth': gdp_growth}
for new_worksheet_name, data in worksheet_name.items():
    if new_worksheet_name in worksheet_list:
        sheet1 = sheet.worksheet(new_worksheet_name)
    else:
        sheet1 = sheet.add_worksheet(title=new_worksheet_name, rows=10, cols=10)
    set_with_dataframe(sheet1, data)

# sheet.worksheet(new_worksheet_name) -- to select worksheet page
# sheet.add_worksheet(new_worksheet_name, rows=10, cols=10) -- add new worksheet page

c:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\NLP Project\cfm-nlp\data.py:161: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"]) + pd.offsets.MonthEnd(0)
c:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\NLP Project\cfm-nlp\data.py:296: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df2["Date"] = pd.to_datetime(df2["Date"]) + pd.offsets.MonthEnd(0)
c:\USERS\AHMADAIZUDEEN\ONEDRIVE - THE SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\DESKTOP\NLP PROJECT\VENC\Lib\site-packages\pandas\core\indexes\base.py:7588: FutureWarning: Dtype inference on a p

[['Name', 'Age'], ['Alice', '25'], ['Bob', '30'], ['Charlie', '40']]


In [ ]:
# 
# Loop through worksheet names and data
for new_worksheet_name, data in worksheet_name.items():
    if new_worksheet_name in worksheet_list:
        sheet = spreadsheet.worksheet(new_worksheet_name)  # Select existing worksheet
    else:
        sheet = spreadsheet.add_worksheet(title=new_worksheet_name, rows=10, cols=10)  # ✅ Use 'spreadsheet' object
    
    # Upload DataFrame to Google Sheets
    set_with_dataframe(sheet, data)

print("All data uploaded successfully!")

AttributeError: 'Worksheet' object has no attribute 'add_worksheet'

In [124]:
import pandas as pd 

yield_data1 = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\00 dlx.xlsx", header=1)
yield_data1 = yield_data1[[*yield_data1.columns[[0,1,3]]]]
yield_data1.drop([0,1,2], inplace=True)
yield_data1.rename(columns={'.DESC': 'Date',
                          'Indonesia: 10 Year Treasury Bond Mid Yield (% p.a.)': 'Indonesia',
                          'Philippines: 10 Year Treasury Bond Mid Yield (% p.a.)': 'Philippines'},
                    inplace=True)

# yield_data1["Date"] = pd.to_datetime(yield_data1["Date"], errors='coerce') + pd.offsets.MonthEnd(0)
yield_data1['Month'] = yield_data1['Date'].apply(lambda x: x[:3])
yield_data1['Year'] = yield_data1['Date'].apply(lambda x: x[-2:])
yield_data1["Date"] = yield_data1.apply(lambda x: pd.to_datetime(f"{x['Month']} {x['Year']}", format='%b %y') + pd.offsets.MonthEnd(0), axis=1)
yield_data1.drop(['Month', 'Year'], axis=1, inplace=True)
yield_data2 = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Capital Flow Monitor\SEACEN CFM January 2025\edited files\Data\Financial Stress Indices.xlsx"
                            , sheet_name='Yields2'
                            , header=1)
yield_data2=yield_data2[yield_data2.columns[:11]]
yield_data2.drop([0,1], inplace=True)
yield_data2.rename(columns={'Region': 'Date'}, inplace=True)
yield_data2["Date"] = pd.to_datetime(yield_data2["Date"]) + pd.offsets.MonthEnd(0)
yield_data2.drop('Philippines', axis=1, inplace=True)

yield_data = yield_data2.join(yield_data1.set_index('Date'), on='Date')
yield_data.set_index('Date', inplace=True)

yield_data_calc = yield_data.apply(lambda x : x - x['United States'], axis =1)
yield_data_calc.drop('United States', axis=1, inplace=True)
yield_data_calc = yield_data_calc.loc[yield_data_calc.index >= "2010-01-31"]
yield_data_calc['Asia'] = yield_data_calc.mean(axis= 1)
yield_data_calc = (yield_data_calc - yield_data_calc.mean()) / yield_data_calc.std()
yield_data_calc

,Thailand,Taiwan,Singapore,South Korea,Malaysia,India,Hong Kong SAR (China),China,Indonesia,Philippines,Asia
Date,,,,,,,,,,,
2010-01-31,0.175651,-1.158241,-1.901368,1.572966,-1.004538,-1.107842,-1.050142,-0.741576,NaN,NaN,-1.540744
2010-02-28,0.087742,-1.236897,-1.485320,1.555688,-0.980961,-0.798749,-1.246005,-0.869708,NaN,NaN,-1.505457
2010-03-31,0.046732,-1.259095,-1.260817,1.023806,-1.166129,-0.865235,-1.426455,-0.900146,NaN,NaN,-1.638070
2010-04-30,-0.370188,-1.407762,-1.865981,0.854675,-1.463067,-1.028513,-1.546308,-1.064175,NaN,NaN,-1.940724
2010-05-31,-0.012640,-0.936574,-0.681719,1.458693,-0.888538,-0.862325,-1.134185,-0.777569,NaN,NaN,-1.390754
...,...,...,...,...,...,...,...,...,...,...,...
2024-07-31,-2.132792,-1.681953,-2.281407,-1.879522,-2.484187,-2.197072,-2.174949,-2.542791,-1.683475,-0.423165,-2.454110
2024-08-31,-1.785504,-1.277460,-1.831526,-1.619181,-1.911162,-1.901352,-1.637959,-2.191315,-1.594144,-0.228112,-2.039476
2024-09-30,-1.675572,-1.240222,-1.707193,-1.427541,-1.779653,-1.885661,-1.585829,-2.078945,-1.606654,-0.250560,-1.949901


In [115]:
yield_data.loc[yield_data.index >= "2010-01-31"].head(30)

,United States,Thailand,Taiwan,Singapore,South Korea,Malaysia,India,Hong Kong SAR (China),China,Indonesia,Philippines
Date,,,,,,,,,,,
2010-01-31,3.733158,4.17,1.51,2.54,5.3535,4.282,7.5814,2.932,3.6986,NaN,NaN
2010-02-28,3.691053,4.05,1.41,2.69,5.297895,4.257,7.857,2.832,3.5111,NaN,NaN
2010-03-31,3.727391,4.05,1.43,2.83,4.918636,4.159,7.825,2.815,3.5129,NaN,NaN
2010-04-30,3.846818,3.8,1.44,2.67,4.905909,4.063,7.7766,2.899,3.4462,NaN,NaN
2010-05-31,3.42,3.69,1.36,2.79,4.951053,4.053,7.5206,2.594,3.3446,NaN,NaN
2010-06-30,3.204091,3.42,1.44,2.37,4.934762,3.957,7.5758,2.336,3.3979,NaN,NaN
2010-07-31,3.011429,3.41,1.4,1.95,4.907727,3.918,7.8286,2.251,3.3357,NaN,NaN
2010-08-31,2.698636,3.31,1.24,1.99,4.676364,3.714,7.9863,1.992,3.3021,NaN,NaN
2010-09-30,2.647619,3.1,1.21,2.02,4.28,3.625,7.8963,2.057,3.4047,7.907477,NaN


In [123]:
yield_data_calc = yield_data.apply(lambda x : x - x['United States'], axis =1)
yield_data_calc.drop('United States', axis=1, inplace=True)
yield_data_calc = yield_data_calc.loc[yield_data_calc.index >= "2010-01-31"]
yield_data_calc['Asia'] = yield_data_calc.mean(axis= 1)
yield_data_calc = (yield_data_calc - yield_data_calc.mean()) / yield_data_calc.std()
yield_data_calc

,Thailand,Taiwan,Singapore,South Korea,Malaysia,India,Hong Kong SAR (China),China,Indonesia,Philippines,Asia
Date,,,,,,,,,,,
2010-01-31,0.175651,-1.158241,-1.901368,1.572966,-1.004538,-1.107842,-1.050142,-0.741576,NaN,NaN,-1.540744
2010-02-28,0.087742,-1.236897,-1.485320,1.555688,-0.980961,-0.798749,-1.246005,-0.869708,NaN,NaN,-1.505457
2010-03-31,0.046732,-1.259095,-1.260817,1.023806,-1.166129,-0.865235,-1.426455,-0.900146,NaN,NaN,-1.638070
2010-04-30,-0.370188,-1.407762,-1.865981,0.854675,-1.463067,-1.028513,-1.546308,-1.064175,NaN,NaN,-1.940724
2010-05-31,-0.012640,-0.936574,-0.681719,1.458693,-0.888538,-0.862325,-1.134185,-0.777569,NaN,NaN,-1.390754
...,...,...,...,...,...,...,...,...,...,...,...
2024-07-31,-2.132792,-1.681953,-2.281407,-1.879522,-2.484187,-2.197072,-2.174949,-2.542791,-1.683475,-0.423165,-2.454110
2024-08-31,-1.785504,-1.277460,-1.831526,-1.619181,-1.911162,-1.901352,-1.637959,-2.191315,-1.594144,-0.228112,-2.039476
2024-09-30,-1.675572,-1.240222,-1.707193,-1.427541,-1.779653,-1.885661,-1.585829,-2.078945,-1.606654,-0.250560,-1.949901


In [36]:
# https://www.seacen.org/publication-working.php?pid=702001-100492 -- kene search
# https://www.seacen.org/publications/RePEc/702001-100492-PDF.pdf -- word

ks = r'https://www.seacen.org/publication-working.php?pid='
wd_list = ['https://www.seacen.org/publications/RePEc/702001-100492-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100483-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100482-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100480-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100471-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100461-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100449-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100444-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-0-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100430-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100427-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100417-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100413-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100402-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100388-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100378-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100334-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100070-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100074-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100071-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100300-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702001-100242-PDF.pdf',
           'https://www.seacen.org/publications/RePEc/702004-100259-PDF.pdf'
           ,
           ]

ks.split('=')[-1]

''

In [38]:
list1 = []
for wd in wd_list:
    wdl = wd.split('/')[-1].split('.')[0].split('-')[:2]
    id = '-'.join(wdl)
    link = ks + id
    list1.append(link)
# ks1 = ks.split('=')[0]
# tuple(zip(id, ks1))

In [39]:
list1

['https://www.seacen.org/publication-working.php?pid=702001-100492',
 'https://www.seacen.org/publication-working.php?pid=702001-100483',
 'https://www.seacen.org/publication-working.php?pid=702001-100482',
 'https://www.seacen.org/publication-working.php?pid=702001-100480',
 'https://www.seacen.org/publication-working.php?pid=702001-100471',
 'https://www.seacen.org/publication-working.php?pid=702001-100461',
 'https://www.seacen.org/publication-working.php?pid=702001-100449',
 'https://www.seacen.org/publication-working.php?pid=702001-100444',
 'https://www.seacen.org/publication-working.php?pid=702001-0',
 'https://www.seacen.org/publication-working.php?pid=702001-100430',
 'https://www.seacen.org/publication-working.php?pid=702001-100427',
 'https://www.seacen.org/publication-working.php?pid=702001-100417',
 'https://www.seacen.org/publication-working.php?pid=702001-100413',
 'https://www.seacen.org/publication-working.php?pid=702001-100402',
 'https://www.seacen.org/publication-wo

In [40]:
import requests as r
for x in list1:
    response = r.get(x)
    print(response.status_code)

200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200


In [ ]:
print(len(list1))
print(len(set(list1))) 
#no duplicate

23
23
